# Meridian GeoX Demo

## Multicell design and analysis

This demo showcases an end-to-end multicell experiment design & analysis using
the library.

-   **Study Design**: Selecting the desired experiment type and using the Google
    design algorithms to generate a list of ranked, viable designs based on your
    specific marketing objectives. Finalize one practical design based on your
    marketing needs and proceed with experiment implementation.
-   **Analysis & Inference**: Analyzing results using counterfactual modeling
    (e.g., time-based regression) and robust inference methods to evaluate
    statistical significance of incremental effect.

## Setup and Installation

In [ ]:
# Install meridian-geox: from PyPI @ latest release
!pip install --upgrade meridian-geox

In [ ]:
import datetime
from pprint import pprint
import warnings
from IPython.display import display
import meridian_geox as geox
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in divide",
    category=RuntimeWarning,
)

pd.set_option("display.max_columns", None)

meridian_geox_root = None

In [ ]:
# If you prefer to save the design in Google Drive, run this cell.
import os
from google.colab import drive

drive_mount = '/content/drive'
drive.mount(drive_mount, force_remount=True)

# Optional: specify a subfolder to organize your runs
subfolder = ''  # @param {"type":"string", "placeholder": "e.g., my_geox_project"}

# Define the root path specifically for GeoX designs and reports
meridian_geox_root = f'{drive_mount}/MyDrive/{subfolder}'

# Create the directory if it doesn't exist
if not os.path.exists(meridian_geox_root):
  os.makedirs(meridian_geox_root)
  print(f'Created directory: {meridian_geox_root}')
else:
  print(f'Using directory: {meridian_geox_root}')

## 1. Multicell design

A Multicell design tests multiple marketing interventions (Treatments) against a
baseline control group (Control) within a single experiment.

### Load data

Generating a design requires at least 3N (where N is the pretest duration) of
pretest data. Please enter your data source link and column names.

-   **date**: `YYYY-MM-DD` string.
-   **location**: e.g., different market areas.
-   **conversion**: Recent conversion data is usually raw, unfiltered
    conversions or revenue data, typically from an advertiser's CRM. Conversion
    data should reflect a business-as-usual (BAU) situation. For multicell,
    conversions must be the overall business conversion metric shared across all
    treatment and control geos.
-   **spend**: spend data for campaigns included in the study. This is an
    optional column. For multi-cell studies, provide separate columns
    (`spend_cell_1`, `spend_cell_2`, etc.) mapped to each cell's specific
    intervention:
    -   Case A: Testing different strategies on the SAME channel/campaign (e.g.,
        Cell 1 = Go-Dark on YouTube; Cell 2 = Heavy-Up on YouTube). Both cells
        modify the same underlying campaign pool. spend_cell_1 and spend_cell_2
        will have identical values (as seen in our example dataset).
    -   Case B: Testing DIFFERENT channels or tactics (e.g., Cell 1 = YouTube
        Ads; Cell 2 = Demand Gen Ads, OR Prospecting vs. Retargeting; and
        another cross-publisher example, Cell 1 = Publisher A spend, Cell 2 =
        Publisher B spend). Each cell modifies a distinct campaign. spend_cell_1
        should contain Tactic A spend, and spend_cell_2 should contain Tactic B
        spend (values will differ).

In [ ]:
# Load the design data
multicell_design_data = pd.read_csv(
    "https://raw.githubusercontent.com/google/meridian-geox/refs/heads/main/meridian_geox/data/example_design_data_multi_cell_go_dark_heavy_up.csv"
)

# Preview the first few rows to ensure it loaded correctly
multicell_design_data.head()

In [ ]:
# Use two cell study as an example.
date_column_name = "date"  # @param {type:"string"}
location_column_name = "location"  # @param {type:"string"}
conversions_column_name = "conversions"  # @param {type:"string"}
spend_cell_1_column_name = "spend_cell_1"  # @param {type:"string"}
spend_cell_2_column_name = "spend_cell_2"  # @param {type:"string"}

multicell_design_data.columns = multicell_design_data.columns.str.lower()

rename_mapping = {
    date_column_name.lower(): "date",
    location_column_name.lower(): "location",
    conversions_column_name.lower(): "conversions",
}

# Make spend column optional
if (
    spend_cell_1_column_name
    and spend_cell_1_column_name.lower() in multicell_design_data.columns
):
  rename_mapping[spend_cell_1_column_name.lower()] = "spend_cell_1"
if (
    spend_cell_2_column_name
    and spend_cell_2_column_name.lower() in multicell_design_data.columns
):
  rename_mapping[spend_cell_2_column_name.lower()] = "spend_cell_2"

# Rename the columns in the dataframe to match the library's expected names
multicell_design_data = multicell_design_data.rename(columns=rename_mapping)

print("Columns mapped successfully!")
display(multicell_design_data.head())

In [ ]:
numeric_cols = ["conversions"]
if "spend_cell_1" in multicell_design_data.columns:
  numeric_cols.append("spend_cell_1")
if "spend_cell_2" in multicell_design_data.columns:
  numeric_cols.append("spend_cell_2")

for colname in numeric_cols:
  multicell_design_data[colname] = pd.to_numeric(multicell_design_data[colname])
multicell_design_data["date"] = pd.to_datetime(multicell_design_data["date"])
multicell_design_data["location"] = multicell_design_data["location"].astype(
    str
)
multicell_design_data.head()

### Generate design

**Clarification of Baseline Spend vs. Required Budget**:

-   **Baseline Spend**: The historical spend in your input dataset.
-   **Required Budget**: The actual spend change for the test:
    -   *Heavy-Up*: Additional budget required above baseline in treatment geos.
    -   *Go-Dark / Go-Dim*: Spend held back / turned off in treatment geos.
    -   *Holdback*: Spend required on the new campaign (MDE_abs x CpIC).

For detailed definitions of the experiment design input parameters, please refer
to the
[API reference](https://developers.google.com/meridian/geox/api-reference#designconfig).
Guidance for a couple of specific fields is provided below.

**cost_per_incremental_conversion (CpIC)**: Equivalent to `1/target iROAS` if
revenue data is used. This is used to estimate budget requirements for a
holdback experiment (cell). For multi-cell experiments, different values can be
assigned per holdback cell using a dictionary. If a single float is provided for
a multi-cell design, it will be applied to all holdback cells. The default is
1.0.

**Budget Constraints**: The budget constraint for the experiment design (per
cell). This could be a total budget amount (eg. `budget=500000`) or a budget
percentage change (eg. `budget_pct=1.0`). For multi-cell experiments, different
values can be assigned per cell using a dictionary; otherwise, the single
provided value is applied to all cells. If the budget percentage change is not
specified for a go-dark cell, the default value is -100%. For a heavy-up cell,
the default value is 100%.

**Exclusions (excluded_geos, excluded_dates)**: Pass the set of geos or dates to
exclude from the design. Remove the field or pass an empty set if there are no
geos or dates to exclude.

**max_conversions_percent**: The maximum conversion volume allowed for the
treatment group. For multi-cell designs, this percentage refers to the total for
all treatment cells. The default is 0.3.

In [ ]:
# Design experiment
print('\nDesigning experiment...')
multicell_experiment_types = {
    'cell_1': geox.ExperimentType.GO_DARK,
    'cell_2': geox.ExperimentType.HEAVY_UP,
}
multicell_design_config = geox.DesignConfig(
    experiment_duration=datetime.timedelta(days=30),
    experiment_types=multicell_experiment_types,
    methodology=geox.Methodology.TBR,
    geo_assignment_rule=geox.GeoAssignmentRule.STRATIFIED_SAMPLING,
    cell_count=2,
    design_output_count=5,
)
multicell_budget_constraint = {
    'cell_1': geox.Budget(budget_pct=-1.0),
    'cell_2': geox.Budget(budget_pct=1.0),
}
multicell_constraints = geox.Constraints(
    excluded_geos={'105'},
    budget_constraint=multicell_budget_constraint,
    max_conversions_percent=0.3,
)

Please indicate if you would like to automatically exclude geos with no response
and outlier dates during design.

If you set this to `False`, you will still see the detected outliers in the
results, allowing you to manually specify any exclusions using the `Constraints`
input above and regenerate your design.

In [ ]:
exclude_geos_no_response = True  # @param {type:"boolean"}
exclude_outlier_dates_design = True  # @param {type:"boolean"}

design_data_quality_check_config = geox.QualityCheckConfig(
    exclude_geos_no_response=exclude_geos_no_response,
    exclude_outlier_dates=exclude_outlier_dates_design,
)

In [ ]:
print("\nDesigning experiment...")
multicell_design_set = geox.run_design(
    multicell_design_data,
    multicell_design_config,
    multicell_constraints,
    design_data_quality_check_config,
)

### Check design results

List the design metrics for all generated designs. Please see the
[API reference](https://developers.google.com/meridian/geox/api-reference#designset)
for the definition of all design metrics.

In [ ]:
multicell_design_set.design_metrics

Enter the selected design ID. Leave this empty to automatically select the
top-ranked design.

In [ ]:
multicell_selected_design_id = ""  # @param {type:"string"}

if not multicell_selected_design_id.strip():
  multicell_selected_design_id = multicell_design_set.design_metrics[
      "design_id"
  ].iloc[0]
  print(
      "Automatically selected top ranked design ID:"
      f" {multicell_selected_design_id}"
  )
else:
  print(f"Manually selected design ID: {multicell_selected_design_id}")

In [ ]:
# Filter the design based on the selected ID
if multicell_selected_design_id in multicell_design_set.designs:
  selected_design = multicell_design_set.designs[multicell_selected_design_id]
  print(f"Successfully selected design: {multicell_selected_design_id}")
  pprint(selected_design)
else:
  print(
      "Error: Design ID not found. Please enter a valid ID from the metrics"
      " table above."
  )

**Evaluating design feasibility (Design implied CpIC):**

Compare the `Implied CpIC` against your `Expected CpIC` to ensure the experiment
is adequately powered.

*   If Implied CpIC < Expected CpIC, your test is underpowered (unlikely to
    detect lift). You must increase budget or test duration.
*   If Implied CpIC ≥ Expected CpIC, your test is adequately powered.

For Holdback study, the design implied CpIC equals to your input CpIC.

In [ ]:
selected_design = multicell_design_set.designs[multicell_selected_design_id]
for cell_name, cell_design in selected_design.designs.items():
  print(f"Design implied CpIC ({cell_name}): {cell_design.design_implied_cpic}")

**Check the data quality and outlier detection results**

In [ ]:
# If the output below shows empty sets, no outlier geos or dates were detected.
selected_design.quality_check_result

In [ ]:
# Show all data quality check results.
if selected_design.quality_check_result is not None:
  design_quality_metrics = selected_design.quality_check_result.quality_metrics
  if design_quality_metrics.empty:
    print("No data quality issues identified.")
  else:
    display(design_quality_metrics)
else:
  print("No quality check result available.")

### Plot design

In [ ]:
geox.plot_design(multicell_design_set.designs[multicell_selected_design_id])

### Save design

In [ ]:
multicell_saved_design_json = multicell_design_set.designs[
    multicell_selected_design_id
].export_to_json()
multicell_saved_design_json

In [ ]:
# Write the JSON string to a file on Google Drive if mounted to Google Drive
if meridian_geox_root:
  multicell_design_file_path = os.path.join(
      meridian_geox_root, "multicell_design.json"
  )
  with open(multicell_design_file_path, "w") as f:
    f.write(multicell_saved_design_json)
  print(
      "Design successfully saved to Google Drive at:"
      f" {multicell_design_file_path}"
  )
else:
  print("Google Drive not mounted. Design exists in session memory only.")

## 2. Multicell analysis

Post-experiment analysis (Run AFTER the experiment has completed).

### Load data and design

Dataset must include historical pretest data and test period data with optional
cooldown period data.

In [ ]:
# Load the analysis data
multicell_analysis_data = pd.read_csv(
    "https://raw.githubusercontent.com/google/meridian-geox/refs/heads/main/meridian_geox/data/example_analysis_data_multi_cell_go_dark_heavy_up.csv"
)
multicell_analysis_data.head()

In [ ]:
date_column_name = "date"  # @param {type:"string"}
location_column_name = "location"  # @param {type:"string"}
conversions_column_name = "conversions"  # @param {type:"string"}
spend_cell_1_column_name = "spend_cell_1"  # @param {type:"string"}
spend_cell_2_column_name = "spend_cell_2"  # @param {type:"string"}

multicell_analysis_data.columns = multicell_analysis_data.columns.str.lower()

rename_mapping = {
    date_column_name.lower(): "date",
    location_column_name.lower(): "location",
    conversions_column_name.lower(): "conversions",
}

# Make spend column optional
if (
    spend_cell_1_column_name
    and spend_cell_1_column_name.lower() in multicell_analysis_data.columns
):
  rename_mapping[spend_cell_1_column_name.lower()] = "spend_cell_1"
if (
    spend_cell_2_column_name
    and spend_cell_2_column_name.lower() in multicell_analysis_data.columns
):
  rename_mapping[spend_cell_2_column_name.lower()] = "spend_cell_2"

# Rename the columns in the dataframe to match the library's expected names
multicell_analysis_data = multicell_analysis_data.rename(columns=rename_mapping)

print("Columns mapped successfully!")
display(multicell_analysis_data.head())

In [ ]:
numeric_cols = ["conversions"]
if "spend_cell_1" in multicell_analysis_data.columns:
  numeric_cols.append("spend_cell_1")
if "spend_cell_2" in multicell_analysis_data.columns:
  numeric_cols.append("spend_cell_2")

for colname in numeric_cols:
  multicell_analysis_data[colname] = pd.to_numeric(
      multicell_analysis_data[colname]
  )
multicell_analysis_data["date"] = pd.to_datetime(
    multicell_analysis_data["date"]
)
multicell_analysis_data["location"] = multicell_analysis_data[
    "location"
].astype(str)
multicell_analysis_data.head()

In [ ]:
design_file_name = "multicell_design.json"  # @param {type:"string"}

# Try to load the design from Google Drive, with a fallback to session memory.
multicell_loaded_design = None

if meridian_geox_root:
  multicell_design_file_path = os.path.join(
      meridian_geox_root, design_file_name
  )
  if os.path.exists(multicell_design_file_path):
    with open(multicell_design_file_path, "r") as f:
      multicell_design_json_text = f.read()
    multicell_loaded_design = geox.Design.load_from_json(
        multicell_design_json_text
    )
    print(
        "Design successfully loaded from Google Drive:"
        f" {multicell_design_file_path}"
    )
  else:
    print(
        f"Design file not found at {multicell_design_file_path}. Will try to"
        " load from session memory."
    )

if multicell_loaded_design is None:
  try:
    multicell_loaded_design = geox.Design.load_from_json(
        multicell_saved_design_json
    )
    print("Design successfully loaded from session memory.")
  except NameError:
    print(
        "Error: Design data not found on Drive or in memory. Please run the"
        " design step first."
    )

### Generate analysis

Please indicate if you would like to automatically remove outlier dates during
the pretest period.

If you set this to `False`, you will still see the detected outliers in the
results, allowing you to manually specify any exclusions using the
`AnalysisConfig` input and regenerate your analysis.

Note that all excluded geos during the design phase will be automatically
removed from the analysis phase.

In [ ]:
exclude_outlier_dates_analysis = True  # @param {type:"boolean"}

analysis_data_quality_check_config = geox.QualityCheckConfig(
    exclude_outlier_dates=exclude_outlier_dates_analysis,
)

In [ ]:
# Analyze experiment results
print("\nAnalyzing experiment results...")
multicell_analysis_config = geox.AnalysisConfig(
    design=multicell_loaded_design,
    analysis_start_date=pd.to_datetime("2020-04-01"),
    analysis_end_date=pd.to_datetime("2020-04-30"),
)
multicell_analysis_result = geox.analyze(
    multicell_analysis_data, multicell_analysis_config
)

### Check analysis result

Please check the
[API reference](https://developers.google.com/meridian/geox/api-reference#analysismetrics)
for the detailed analysis result definition.

iCPD is equivalent to iROAS if revenue data is used. It is populated if spend
data is available.

A geo experiment is considered statistically inconclusive when its confidence
interval includes zero.

In [ ]:
print(f"Analysis result:")
pprint(multicell_analysis_result.results)

In [ ]:
# Show first 5 days of the time series data.
multicell_analysis_result.results["cell_1"].cumulative_lift.head(5)

In [ ]:
multicell_analysis_result.results["cell_2"].cumulative_lift.head(5)

In [ ]:
if "spend_cell_1" in multicell_analysis_data.columns:
  display(multicell_analysis_result.results["cell_1"].cumulative_icpd.head(5))
else:
  print(
      "Cumulative iCPD is not available because the 'spend_cell_1' column was"
      " not provided."
  )

In [ ]:
if "spend_cell_2" in multicell_analysis_data.columns:
  display(multicell_analysis_result.results["cell_2"].cumulative_icpd.head(5))
else:
  print(
      "Cumulative iCPD is not available because the 'spend_cell_2' column was"
      " not provided."
  )

**Check the data quality and outlier detection results**

In [ ]:
# If the output below shows empty sets, no outlier geos or dates were detected.
multicell_analysis_result.quality_check_result

In [ ]:
if multicell_analysis_result.quality_check_result is not None:
  analysis_quality_metrics = (
      multicell_analysis_result.quality_check_result.quality_metrics
  )
  if analysis_quality_metrics.empty:
    print("No data quality issues identified.")
  else:
    display(analysis_quality_metrics)
else:
  print("No quality check result available.")

### Plot analysis

In [ ]:
geox.plot_analysis(multicell_analysis_result)

## 3. Compare designs

Single function call to test multiple `DesignConfig` setups (e.g., Random vs.
Stratified Sampling, or 14-day vs. 30-day duration) on the same dataset
side-by-side.

In [ ]:
comparison_design_results = geox.compare_designs(
    multicell_design_data,
    [
        (
            geox.DesignConfig(
                experiment_duration=datetime.timedelta(days=30),
                experiment_types={
                    'cell_1': geox.ExperimentType.GO_DARK,
                    'cell_2': geox.ExperimentType.HEAVY_UP,
                },
                methodology=geox.Methodology.TBR,
                geo_assignment_rule=geox.GeoAssignmentRule.RANDOM,
                cell_count=2,
            ),
            geox.Constraints(),
        ),
        (
            geox.DesignConfig(
                experiment_duration=datetime.timedelta(days=30),
                experiment_types={
                    'cell_1': geox.ExperimentType.GO_DARK,
                    'cell_2': geox.ExperimentType.HEAVY_UP,
                },
                methodology=geox.Methodology.TBR,
                geo_assignment_rule=geox.GeoAssignmentRule.STRATIFIED_SAMPLING,
                cell_count=2,
            ),
            geox.Constraints(),
        ),
    ],
)

In [ ]:
# Best design (by MDE)
comparison_selected_design_id = next(iter(comparison_design_results.designs))
pprint(comparison_design_results.designs[comparison_selected_design_id])

In [ ]:
comparison_design_results.design_metrics

## 4. Concatenate two designs

Combines pre-existing `DesignSet` objects into a single combined ranking table.

In [ ]:
# Generate two multicell designs
print("\nGenerating first multicell design...")
design_config_1 = geox.DesignConfig(
    experiment_duration=datetime.timedelta(days=30),
    experiment_types={
        "cell_1": geox.ExperimentType.GO_DARK,
        "cell_2": geox.ExperimentType.HEAVY_UP,
    },
    methodology=geox.Methodology.TBR,
    geo_assignment_rule=geox.GeoAssignmentRule.RANDOM,
    cell_count=2,
)
design_set_1 = geox.run_design(
    multicell_design_data,
    design_config_1,
    constraints=geox.Constraints(),
)
print("First multicell design generated.")
design_set_1.design_metrics

In [ ]:
print("\nGenerating second multicell design...")
design_config_2 = geox.DesignConfig(
    experiment_duration=datetime.timedelta(days=30),
    experiment_types={
        "cell_1": geox.ExperimentType.GO_DARK,
        "cell_2": geox.ExperimentType.HEAVY_UP,
    },
    methodology=geox.Methodology.TBR,
    geo_assignment_rule=geox.GeoAssignmentRule.STRATIFIED_SAMPLING,
    cell_count=2,
)
design_set_2 = geox.run_design(
    multicell_design_data, design_config_2, constraints=geox.Constraints()
)
print("Second multicell design generated.")
design_set_2.design_metrics

In [ ]:
# Concatenate the two newly generated design sets
print("\nConcatenating the two multicell design sets...")
combined_design_set = geox.concat_design_reports([design_set_1, design_set_2])

print("Combined design metrics (ranked by mde):")
combined_design_set.design_metrics